In [6]:
from utils import *  # Helper classes for reading colorterm data and FLOWSql

#### Set the FLOWS_API_KEY variable appropriately,
#### this hack may not work unless FLOWS_API_KEY is set in the bash environment ####
FLOWS_API_KEY=!bash -i -c 'echo $FLOWS_API_KEY'
FLOWS_API_KEY=FLOWS_API_KEY[-1]

In [7]:
#Reading colorterm results from file

colorterms=IncrementalStruct()
colorterms.loadJsons('colorterm_results.txt');

In [38]:
from astropy.table import Table
from astropy.stats import SigmaClip, mad_std
import numpy as np

flt_map={
    'u_mag': 'up',
    'g_mag': 'gp',
    'r_mag': 'rp',
    'i_mag': 'ip',
    'z_mag': 'zp',
}

basedir='/nerdrage/calc/archive/photometry/'

overwrite=False

In [40]:
fsql = FLOWSql(FLOWS_API_KEY)

if overwrite:
    sel_cond = ""
else:
    sel_cond = "and ps.mag_ctcor is null"

for sitename in colorterms().keys():
    print(sitename)

    lc_data=[]
    fetch_limit=5000
    last_fetch=fetch_limit
    while last_fetch==fetch_limit and len(lc_data)<100_000:  # Hard limit of 100k to avoid any misconfiguration causing infinite API queries 
        buff=fsql.query("""
        select * from photometry_summary ps
        join files f on f.fileid = ps.fileid_phot
        join sites s on f.site = s.siteid
        join targets t ON f.targetid = t.targetid
        where s.sitename='"""+sitename+"""'
        and ps.photfilter in """+'('+','.join(["'"+flt_map.get(flt,flt.replace('_mag',''))+"'" for flt in colorterms()[sitename].keys()])+')'+"""
        """+sel_cond+"""
        limit """+str(fetch_limit)+""" offset """+str(len(lc_data))+"""
        """)
        last_fetch=buff['count']
        lc_data.extend(buff['rows'])
        print(len(lc_data))

    for row in lc_data:
        mag_col = row['photfilter'][0:1]+'_mag'
        color_col0, color_col1 = colorterms()[sitename][mag_col]['color']
        color_term = colorterms()[sitename][mag_col]['colorterm']

        print(f"Processing {row['target_name']}, fileid: {row['fileid_img']}")
        
        t = Table.read(basedir + row['path'], format='ascii.ecsv')
        SNt= t[t['starid']<=0] # SN mags only
        t = t[t['starid']>0]  # skip SN mag

        # Required columns
        needed = ['flux_psf', 'flux_psf_error', 'mag', 'mag_error',
                    mag_col, color_col0, color_col1]
        if not all(col in t.colnames for col in needed):
            missing = [c for c in needed if c not in t.colnames]
            print(f"  Skipping {filepath}: missing columns {missing}")
            continue

        flux     = np.array(t['flux_psf'].data,        dtype=float)
        flux_err = np.array(t['flux_psf_error'].data,  dtype=float)
        mag      = np.array(t['mag'].data,             dtype=float)
        ref_mag  = np.array(t[mag_col].data,           dtype=float)
        col0     = np.array(t[color_col0].data,        dtype=float)
        col1     = np.array(t[color_col1].data,        dtype=float)

        # Mask out bad values
        valid = (flux > 0) & np.isfinite(flux) & np.isfinite(ref_mag) \
                & np.isfinite(col0) & np.isfinite(col1) & (ref_mag !=0) & (col0 !=0) & (col1 !=0) & (col0 != col1)
        flux     = flux[valid];     flux_err = flux_err[valid]
        mag      = mag[valid];      ref_mag  = ref_mag[valid]
        col0     = col0[valid];     col1     = col1[valid]

        if not valid.any():
            print(f"  Skipping {basedir + row['path']} {row['photfilter']}, : no valid data")
            continue

        # Instrumental magnitude and error
        instmag       = -2.5 * np.log10(flux) + 24
        instmag_error = 2.5 * flux_err / flux / np.log(10)

        # Axes
        Dmag = mag - ref_mag                  # instrumental - catalog mag
        colormag = col0 - col1                        # e.g. J_mag - H_mag

        if np.any(colormag==0):
            print(f"  colormag==0 for {basedir+filepath}")


        tmp = Dmag - color_term[0] * colormag
        sigclip = SigmaClip(sigma=3, maxiters=5)
        mask = ~sigclip(tmp).mask
        zp = [np.median(tmp[mask]), np.std(tmp[mask]) ]
        #CP_all_Dmag_err[i] = np.sqrt( CP_all_Dmag_err[i]**2 + (np.std(tmp[mask]))**2 )

        #SNt.pprint(max_width=-1)

        SNt['mag_ctcor']= np.array(SNt['mag'].data, dtype=float)+zp[0]
        SNt['mag_ctcor_error']= np.sqrt( (2.5* np.array(SNt['flux_psf_error'].data,dtype=float)/np.array(SNt['flux_psf'].data,dtype=float) /np.log(10) )**2 +  zp[1]**2)
        #print(zp)
        #SNt.pprint(max_width=-1)
        #SNt[SNt['starid']==0]['mag','mag_error']
        #SNt['mag','mag_error', 'mag_ccor', 'mag_ccor_error'].pprint()
        #break
        fsql.query("update photometry_summary set mag_ctcor="+str(SNt['mag_ctcor'][0])+", mag_ctcor_error="+str(SNt['mag_ctcor_error'][0])+" where fileid_phot="+str(row['fileid_phot']), admin=True)


NOT
1196
Processing 2019muj, fileid: 1023
Processing 2019muj, fileid: 1015


Processing 2019yvr, fileid: 318
Processing 2019muj, fileid: 1016
Processing 2019muj, fileid: 1014
Processing 2019muj, fileid: 1013
Processing 2019muj, fileid: 1017
Processing 2019muj, fileid: 1021
Processing 2019muj, fileid: 1025
Processing 2019muj, fileid: 1028
Processing 2019muj, fileid: 1008
Processing 2019yvr, fileid: 454
Processing 2019muj, fileid: 1027
Processing 2019yvr, fileid: 455
Processing 2019yvr, fileid: 456
Processing 2022ihz, fileid: 14966
Processing 2019muj, fileid: 1026
Processing 2019muj, fileid: 1033
Processing 2019muj, fileid: 1042
Processing 2019muj, fileid: 1043
Processing 2019muj, fileid: 1047
Processing 2019muj, fileid: 1048
Processing 2019muj, fileid: 1046
Processing 2019muj, fileid: 1041
Processing 2019muj, fileid: 1010
Processing 2019muj, fileid: 1011
Processing 2019muj, fileid: 1018
Processing 2019muj, fileid: 1019
Processing 2019yvr, fileid: 457
Processing 2019muj, fileid: 1032
Processing 2019muj, fileid: 1036
Processing 2019muj, fileid: 1035
Processing 201

### All below is trash

In [21]:
SNt['mag_ctcor'][0]

np.float64(17.081530124741597)

In [39]:
print(row)

{'fileid_img': 357988, 'targetid': 4875, 'obstime': '60692.178247021686', 'photfilter': 'J', 'mag_raw': '16.63988504289774', 'mag_raw_error': '0.05777484778567478', 'photid': 11684, 'pipeline_version': 'v1.2.2', 'fileid_template': None, 'latest_version': 1, 'fileid_phot': 361576, 'mag_sub': None, 'mag_sub_error': None, 'fileid_diffimg': None, 'fileid': 361576, 'path': '2025dd/357988/v01/photometry-2025dd-357988-v01.ecsv', 'archive': 11, 'datatype': 2, 'site': 5, 'filesize': 7428, 'filehash': 'ce8266a8813b6448f92ffb0027b97e1356a33c7c', 'inserted': '2025-01-14 18:03:32.978799', 'lastmodified': '2026-04-01 14:26:16.849208', 'available': 1, 'exptime': None, 'version': 1, 'newest_version': True, 'siteid': 5, 'sitename': 'NOT', 'longitude': '-17.88508', 'latitude': '28.75728', 'elevation': '2382', 'site_keyword': None, 'target_name': '2025dd', 'ra': '162.785926', 'decl': '-2.3498012', 'catalog_downloaded': True, 'discovery_mag': '19.198', 'pointing_model_created': False, 'redshift': '0.08564

In [55]:
SNt[SNt['starid']==0]['mag','mag_error']

mag,mag_error
float64,float64
15.533151222118155,0.08212700463767733


In [43]:
SNt.pprint(max_width=-1)

starid       ra           decl       pm_ra    pm_dec  gaia_mag gaia_bp_mag gaia_rp_mag gaia_variability B_mag V_mag H_mag J_mag K_mag u_mag g_mag r_mag i_mag z_mag distance    pixel_column       pixel_row     used_for_epsf   flux_aperture   flux_aperture_error      flux_psf       flux_psf_error   pixel_column_psf_fit pixel_row_psf_fit pixel_column_psf_fit_error pixel_row_psf_fit_error        mag              mag_error     
            deg           deg       mas / yr mas / yr                                                                                                                 deg                                                                                                                                                                                                                                                                 
------ ------------- -------------- -------- -------- -------- ----------- ----------- ---------------- ----- ----- ----- ----- ----- ----- ----- 

In [5]:
[basedir+lc['path'] for lc in lc_data]

['/nerdrage/calc/archive/photometry/2019yvr/00318/v03/photometry-2019yvr-00318-v03.ecsv',
 '/nerdrage/calc/archive/photometry/2019muj/01016/v06/photometry-2019muj-01016-v06.ecsv',
 '/nerdrage/calc/archive/photometry/2019muj/01015/v06/photometry-2019muj-01015-v06.ecsv',
 '/nerdrage/calc/archive/photometry/2019muj/01014/v06/photometry-2019muj-01014-v06.ecsv',
 '/nerdrage/calc/archive/photometry/2019muj/01013/v06/photometry-2019muj-01013-v06.ecsv',
 '/nerdrage/calc/archive/photometry/2019muj/01023/v06/photometry-2019muj-01023-v06.ecsv',
 '/nerdrage/calc/archive/photometry/2019muj/01017/v06/photometry-2019muj-01017-v06.ecsv',
 '/nerdrage/calc/archive/photometry/2019muj/01021/v06/photometry-2019muj-01021-v06.ecsv',
 '/nerdrage/calc/archive/photometry/2019yvr/00454/v02/photometry-2019yvr-00454-v02.ecsv',
 '/nerdrage/calc/archive/photometry/2019yvr/00455/v02/photometry-2019yvr-00455-v02.ecsv',
 '/nerdrage/calc/archive/photometry/2019yvr/00456/v02/photometry-2019yvr-00456-v02.ecsv',
 '/nerdrag

In [48]:
lc_data[1]

{'fileid_img': 1016,
 'targetid': 4,
 'obstime': '58786.08247673379',
 'photfilter': 'B',
 'mag_raw': '20.105737195720273',
 'mag_raw_error': '0.017256427166129695',
 'photid': 428,
 'pipeline_version': 'v1.2.2',
 'fileid_template': None,
 'latest_version': 6,
 'fileid_phot': 354868,
 'mag_sub': None,
 'mag_sub_error': None,
 'fileid_diffimg': None,
 'fileid': 354868,
 'path': '2019muj/01016/v06/photometry-2019muj-01016-v06.ecsv',
 'archive': 11,
 'datatype': 2,
 'site': 5,
 'filesize': 5470,
 'filehash': '510dce78ee507b5a3c649165c8bfbff2e6bb568d',
 'inserted': '2026-01-03 11:35:54.91067',
 'lastmodified': '2026-01-03 11:35:54.91067',
 'available': 1,
 'exptime': None,
 'version': 6,
 'newest_version': True,
 'siteid': 5,
 'sitename': 'NOT',
 'longitude': '-17.88508',
 'latitude': '28.75728',
 'elevation': '2382',
 'site_keyword': None}

In [22]:
colorterms()

{'NOT': {'g_mag': {'color': ['g_mag', 'r_mag'],
   'colorterm': [0.10412163376124464, 0.0024501103375298367]},
  'timestamp': '2026-04-04, 02:02:07',
  'i_mag': {'color': ['r_mag', 'i_mag'],
   'colorterm': [-0.06977686723925433, 0.0023825217563560037]},
  'r_mag': {'color': ['g_mag', 'r_mag'],
   'colorterm': [0.001460833553429767, 0.0017657596150875159]},
  'u_mag': {'color': ['u_mag', 'g_mag'],
   'colorterm': [-0.04790960682142542, 0.010362597082462892]},
  'B_mag': {'color': ['B_mag', 'V_mag'],
   'colorterm': [-0.04946535286764636, 0.008779178782144932]},
  'V_mag': {'color': ['B_mag', 'V_mag'],
   'colorterm': [0.06634576478850293, 0.006798953321228585]},
  'J_mag': {'color': ['J_mag', 'H_mag'],
   'colorterm': [-0.18710903857443223, 0.0058421773409942385]},
  'H_mag': {'color': ['J_mag', 'H_mag'],
   'colorterm': [0.31428364348101034, 0.007735226257896536]},
  'K_mag': {'color': ['H_mag', 'K_mag'],
   'colorterm': [0.4792791536931, 0.026000555470541795]},
  'z_mag': {'color': [